In [1]:
# Setup
%load_ext autoreload
%autoreload 2

import logging
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from config import Config
from macro_data import DataWrapper
from macro_data.readers.economic_data.exchange_rates import ExchangeRatesReader
from src import distribution_validation as dv

logging.getLogger().setLevel(logging.WARNING)


In [2]:
# Load config and output paths
cfg = Config().from_env()
raw_data_path = cfg.raw_data_path or Path('data/hfcs/raw')
output_dir = cfg.output_path or Path('data/output')
output_dir.mkdir(parents=True, exist_ok=True)

data_pkl_path = output_dir / 'data.pkl'
model_h5_path = output_dir / f"simulation_{cfg.country_iso3}.h5"
if not model_h5_path.exists():
    fallback_h5_path = output_dir / 'multi_country_simulation.h5'
    if fallback_h5_path.exists():
        model_h5_path = fallback_h5_path


In [3]:
def country_entry(mapping, country_iso3):
    if country_iso3 in mapping:
        return mapping[country_iso3]
    for key, value in mapping.items():
        key_value = getattr(key, 'value', key)
        if key_value == country_iso3:
            return value
    raise KeyError(f"Country {country_iso3!r} not found. Available keys: {list(mapping.keys())}")


def timestep_for_period(start_year, start_quarter, year, quarter, time_unit):
    periods_per_year = int(round(12 / time_unit)) if time_unit else 4
    return (year - start_year) * periods_per_year + (quarter - start_quarter)


def national_accounts_value_for_period(national_accounts, year, quarter, column):
    period_index = national_accounts.index.to_period('Q')
    period = pd.Period(year=year, quarter=quarter, freq='Q')
    values = national_accounts.loc[period_index == period, column]
    if values.empty:
        raise KeyError(f'{column!r} not found for {year}Q{quarter} in national accounts.')
    value = float(values.iloc[0])
    if not np.isfinite(value) or value <= 0:
        raise ValueError(f'{column!r} must be positive and finite for {year}Q{quarter}.')
    return value


def timestep_indices_for_year(start_year, start_quarter, year, time_unit, n_steps):
    periods_per_year = int(round(12 / time_unit)) if time_unit else 4
    indices = [
        timestep_for_period(start_year, start_quarter, year, period + 1, time_unit)
        for period in range(periods_per_year)
    ]
    if any(index < 0 or index >= n_steps for index in indices):
        raise ValueError(f'Model output does not contain a full year for {year}.')
    return indices


def model_annual_household_values(values, year, *, scale=1.0, drop_nonpositive=False):
    indices = timestep_indices_for_year(start_year, start_quarter, year, time_unit, np.asarray(values).shape[0])
    annual_values = np.asarray(values, dtype=float)[indices].sum(axis=0) * scale
    return clean_values(annual_values, positive=drop_nonpositive)


def model_annual_gdp(year):
    indices = timestep_indices_for_year(start_year, start_quarter, year, time_unit, len(gdp))
    return float(np.sum(gdp[indices])) * unit_scale


def national_accounts_annual_value(national_accounts, year, column):
    return sum(national_accounts_value_for_period(national_accounts, year, quarter, column) for quarter in range(1, 5))


def clean_values(values, *, positive=False):
    array = np.asarray(values, dtype=float).ravel()
    array = array[np.isfinite(array)]
    return array[array > 0] if positive else array


def gdp_ratio_labels(values):
    return [f'{value:.2f}x GDP' for value in values]



SOURCE_COLORS = {
    'Model': '#1f77b4',
    'HFCS': '#d62728',
}




### Distribution of income and wealth against HFCS


In [4]:
if not data_pkl_path.exists():
    raise FileNotFoundError(f"Processed model data not found: {data_pkl_path}")
if not model_h5_path.exists():
    raise FileNotFoundError(f"Simulation output not found: {model_h5_path}")

data = DataWrapper.init_from_pickle(data_pkl_path)
synthetic_country = country_entry(data.synthetic_countries, cfg.country_iso3)
population = synthetic_country.population
national_accounts = synthetic_country.exogenous_data.national_accounts

model_scale = float(getattr(population, 'scale', getattr(synthetic_country, 'scale', 1.0)))
unit_scale = 1.0 / model_scale
start_year = int(getattr(data.configuration, 'year', getattr(population, 'year', 2014)))
start_quarter = int(getattr(data.configuration, 'quarter', 1) or 1)
time_unit = int(getattr(data.configuration, 'time_unit', getattr(data, 'time_unit', 3)) or 3)

with h5py.File(model_h5_path, 'r') as f:
    country = f[cfg.country_iso3]
    households = country['households']
    economy = country['economy']

    income = households['income'][:]
    wealth = households['wealth'][:]
    wealth_deposits = households['wealth_deposits'][:]
    gdp = economy['gdp_expenditure'][:].ravel()

model_series = {
    'Income': income,
    'Wealth': wealth,
    'Liquid Wealth': wealth_deposits,
    'Illiquid Wealth': wealth - wealth_deposits,
}

hfcs_years = [2014, 2017, 2021]
hfcs_quarter = 1
exchange_rates = ExchangeRatesReader.from_csv(raw_data_path / 'exchange_rates' / 'exchange_rates.csv')
hfcs_wave_frames = dv.load_hfcs_wave_dataframes(
    raw_data_path / 'hfcs',
    cfg.country_iso2,
    hfcs_years,
    eur_to_lcu_by_year={year: exchange_rates.from_eur_to_lcu(cfg.country_iso3, year) for year in hfcs_years},
)

hfcs_wave_series = {}
for year, frame in hfcs_wave_frames.items():
    hfcs_wave_series[year] = {
        'Income': frame['Income'].to_numpy(dtype=float),
        'Wealth': frame['Wealth'].to_numpy(dtype=float),
        'Liquid Wealth': frame['Wealth in Deposits'].to_numpy(dtype=float),
        'Illiquid Wealth': (frame['Wealth'] - frame['Wealth in Deposits']).to_numpy(dtype=float),
    }

pd.DataFrame(
    {
        'item': ['simulation output', 'processed model data', 'start year', 'start quarter', 'time unit months', 'scale', 'HFCS waves'],
        'value': [str(model_h5_path), str(data_pkl_path), start_year, start_quarter, time_unit, model_scale, ', '.join(map(str, hfcs_years))],
    }
)


KeyError: "Unable to synchronously open object (object 'wealth_deposits' doesn't exist)"

In [ ]:
time_index = np.arange(income.shape[0])

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=('Mean household income', 'Mean household wealth'),
)
fig.add_trace(
    go.Scatter(x=time_index, y=np.nanmean(income, axis=1) * unit_scale, mode='lines', name='Income'),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(x=time_index, y=np.nanmean(wealth, axis=1) * unit_scale, mode='lines', name='Wealth'),
    row=1,
    col=2,
)
fig.update_layout(height=420, template='plotly_white', showlegend=False)
fig.update_xaxes(title_text='Simulation step')
fig.update_yaxes(tickformat=',.4~f')
fig.show()


In [ ]:
# Comparison settings
trim_percentile = 99.0
n_bins = 80
hfcs_comparison_steps = {
    year: timestep_for_period(start_year, start_quarter, year, hfcs_quarter, time_unit)
    for year in hfcs_years
}
hfcs_comparison_steps = {
    year: step_idx
    for year, step_idx in hfcs_comparison_steps.items()
    if 0 <= step_idx < income.shape[0]
}
if not hfcs_comparison_steps:
    raise ValueError('No HFCS comparison years fall inside the simulated time span.')

model_annual_gdp_by_year = {
    year: model_annual_gdp(year)
    for year in hfcs_comparison_steps
}
hfcs_annual_gdp_by_year = {
    year: national_accounts_annual_value(national_accounts, year, 'GDP (Value)')
    for year in hfcs_comparison_steps
}

model_name = 'Model'
data_name = 'HFCS'
value_unit_label = 'euro'
comparison_year_label = f"{min(hfcs_comparison_steps)}-{max(hfcs_comparison_steps)}"


In [ ]:
distribution_figures = {}

for label in ('Income', 'Wealth'):
    positive_only = label == 'Wealth'
    display_label = 'Annual Income' if label == 'Income' else label
    series_by_name = {}

    for year, step_idx in hfcs_comparison_steps.items():
        if label == 'Income':
            model_values = model_annual_household_values(model_series[label], year, scale=unit_scale)
        else:
            model_values = dv.prepare_model_values(
                model_series[label],
                timestep=step_idx,
                scale=unit_scale,
                drop_nonpositive=positive_only,
            )
        hfcs_values = hfcs_wave_series[year][label]
        hfcs_values = clean_values(hfcs_values, positive=positive_only)

        series_by_name[f'{model_name} {year}'] = model_values
        series_by_name[f'{data_name} {year}'] = hfcs_values

    trimmed_series, upper_limit = dv.trim_series_to_common_percentile(
        series_by_name,
        percentile=trim_percentile,
        lower_bound=0.0 if positive_only else None,
    )
    title_suffix = f'(HFCS {comparison_year_label}; common p{trim_percentile:g} cutoff: {upper_limit:,.0f})'
    distribution_figures[label] = {
        'overlaid_histogram': dv.build_multi_histogram_figure(
            trimmed_series,
            title=f'{display_label}: model vs HFCS histogram {title_suffix}',
            xaxis_title=f'{display_label} ({value_unit_label})',
            nbinsx=n_bins,
        ),
        'cdf': dv.build_multi_cdf_figure(
            trimmed_series,
            title=f'{display_label}: model vs HFCS CDF {title_suffix}',
            xaxis_title=f'{display_label} ({value_unit_label})',
        ),
    }
    for fig in distribution_figures[label].values():
        fig.show()

income_ratio_rows = []
for year, step_idx in hfcs_comparison_steps.items():
    model_income_total = float(np.sum(model_annual_household_values(income, year, scale=unit_scale)))
    hfcs_income_total = dv.weighted_hfcs_total(hfcs_wave_frames[year], 'Income')
    income_ratio_rows.extend(
        [
            {'year': year, 'source': model_name, 'income_to_gdp': model_income_total / model_annual_gdp_by_year[year]},
            {'year': year, 'source': data_name, 'income_to_gdp': hfcs_income_total / hfcs_annual_gdp_by_year[year]},
        ]
    )
income_gdp_ratios = pd.DataFrame(income_ratio_rows)

income_gdp_fig = go.Figure()
for source, source_df in income_gdp_ratios.groupby('source', sort=False):
    income_gdp_fig.add_trace(
        go.Bar(
            x=source_df['year'].astype(str),
            y=source_df['income_to_gdp'],
            name=source,
            text=gdp_ratio_labels(source_df['income_to_gdp']),
            textposition='outside',
            cliponaxis=False,
            marker_color=SOURCE_COLORS[source],
            hovertemplate=f'{source}<br>HFCS year %{{x}}<br>Annual income / annual GDP: %{{y:.3f}}<extra></extra>',
        )
    )
income_gdp_fig.update_layout(
    title_text='Aggregate annual household income / annual GDP',
    template='plotly_white',
    barmode='group',
    height=420,
)
income_gdp_fig.update_xaxes(title_text='HFCS year')
income_gdp_fig.update_yaxes(tickformat=',.2~f')
income_gdp_fig.show()

income_gdp_ratios


### End-of-simulation income and wealth against pooled HFCS


In [ ]:
periods_per_year = int(round(12 / time_unit)) if time_unit else 4
last_possible_year = start_year + int(np.ceil((start_quarter - 1 + income.shape[0]) / periods_per_year))
complete_model_years = []
for year in range(start_year, last_possible_year + 1):
    try:
        timestep_indices_for_year(start_year, start_quarter, year, time_unit, income.shape[0])
    except ValueError:
        continue
    complete_model_years.append(year)
if not complete_model_years:
    raise ValueError('Model output does not contain one complete calendar year for annual income comparison.')

end_income_year = max(complete_model_years)
end_step_idx = income.shape[0] - 1
pooled_hfcs_frame = pd.concat(hfcs_wave_frames.values(), ignore_index=True)
pooled_hfcs_series = {
    'Income': pooled_hfcs_frame['Income'].to_numpy(dtype=float),
    'Wealth': pooled_hfcs_frame['Wealth'].to_numpy(dtype=float),
}

end_sim_series = {
    'Income': {
        f'{model_name} final year {end_income_year}': model_annual_household_values(
            income,
            end_income_year,
            scale=unit_scale,
        ),
        f'{data_name} pooled {comparison_year_label}': clean_values(pooled_hfcs_series['Income']),
    },
    'Wealth': {
        f'{model_name} final step {end_step_idx}': dv.prepare_model_values(
            wealth,
            timestep=end_step_idx,
            scale=unit_scale,
            drop_nonpositive=True,
        ),
        f'{data_name} pooled {comparison_year_label}': clean_values(pooled_hfcs_series['Wealth'], positive=True),
    },
}

end_sim_figures = {}
for label, series_by_name in end_sim_series.items():
    positive_only = label == 'Wealth'
    display_label = 'Annual Income' if label == 'Income' else label
    trimmed_series, upper_limit = dv.trim_series_to_common_percentile(
        series_by_name,
        percentile=trim_percentile,
        lower_bound=0.0 if positive_only else None,
    )
    title_suffix = f'(pooled HFCS {comparison_year_label}; common p{trim_percentile:g} cutoff: {upper_limit:,.0f})'
    end_sim_figures[label] = {
        'overlaid_histogram': dv.build_multi_histogram_figure(
            trimmed_series,
            title=f'End-of-simulation {display_label}: model vs pooled HFCS histogram {title_suffix}',
            xaxis_title=f'{display_label} ({value_unit_label})',
            nbinsx=n_bins,
        ),
        'cdf': dv.build_multi_cdf_figure(
            trimmed_series,
            title=f'End-of-simulation {display_label}: model vs pooled HFCS CDF {title_suffix}',
            xaxis_title=f'{display_label} ({value_unit_label})',
        ),
    }
    for fig in end_sim_figures[label].values():
        fig.show()

end_sim_summary = []
for label, series_by_name in end_sim_series.items():
    for source, values in series_by_name.items():
        cleaned = clean_values(values, positive=(label == 'Wealth'))
        end_sim_summary.append(
            {
                'metric': 'Annual Income' if label == 'Income' else label,
                'source': source,
                'observations': len(cleaned),
                'mean': float(np.mean(cleaned)),
                'median': float(np.median(cleaned)),
                'p90': float(np.percentile(cleaned, 90)),
                'p99': float(np.percentile(cleaned, 99)),
            }
        )
pd.DataFrame(end_sim_summary)


### Liquid and illiquid wealth Lorenz curves


In [ ]:
asset_labels = ('Liquid Wealth', 'Illiquid Wealth')

asset_lorenz_figures = {}

for label in asset_labels:
    lorenz_series_by_name = {}

    for year, step_idx in hfcs_comparison_steps.items():
        model_asset = dv.prepare_model_values(model_series[label], timestep=step_idx, scale=unit_scale, drop_nonpositive=True)
        hfcs_asset = clean_values(hfcs_wave_series[year][label], positive=True)

        if len(model_asset) == 0 or len(hfcs_asset) == 0:
            raise ValueError(f'{label} must contain positive total wealth for both model and HFCS in {year}.')

        lorenz_series_by_name[f'{model_name} {year}'] = model_asset
        lorenz_series_by_name[f'{data_name} {year}'] = hfcs_asset

    asset_lorenz_figures[label] = dv.build_multi_lorenz_figure(
        lorenz_series_by_name,
        title=f'{label}: model vs HFCS Lorenz curve ({comparison_year_label})',
    )

for fig in asset_lorenz_figures.values():
    fig.show()


### Distribution of MPCs


### Impulse response functions
